# Análisis Exploratorio de Datos (EDA) — Afiliados al SNP

**Curso:** Big Data — UNAJ  
**Docente:** Leibnihtz Abrahan Ayamamani Choque  
**Semestre:** 2026-2  
**Dataset:** Base de Datos de Accidentes de EEUU (2016 - 2023)  
**Fuente:** Sobhan Moosavi - Kaggle  
**Tamaño:** ~2.84 GB (CSV)

---

## Objetivos del Notebook

1. **Cargar y perfilar** un dataset real de escala Big Data (~424 MB)
2. **Decodificar** variables categóricas codificadas numéricamente
3. **Explorar** distribuciones, correlaciones y patrones
4. **Exportar** visualizaciones para la presentación de clase

---

## 0. Setup e Importaciones

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import warnings
import time

warnings.filterwarnings('ignore')

# Estilo global
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 150,
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.3
})

# Paleta del curso
NAVY      = '#1A3C5E'
STEEL     = '#2E86AB'
TEAL      = '#0E6B8B'
GOLD      = '#F4B942'
CORAL     = '#E55B4D'
GREEN     = '#27AE60'
LIGHT_BG  = '#F5F9FC'
PALETTE   = [STEEL, NAVY, GOLD, CORAL, GREEN, TEAL, '#8E44AD', '#E67E22']

# Directorio de salida para gráficos
OUTPUT_DIR = Path('../outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(name: str, fig=None):
    """Guarda el gráfico actual en outputs/ como PNG de alta resolución."""
    path = OUTPUT_DIR / f'{name}.png'
    if fig:
        fig.savefig(path, dpi=200, bbox_inches='tight', facecolor='white')
    else:
        plt.savefig(path, dpi=200, bbox_inches='tight', facecolor='white')
    print(f'  ✅ Guardado: {path}')

print('✅ Librerías cargadas')
print(f'   Pandas {pd.__version__} | NumPy {np.__version__}')

✅ Librerías cargadas
   Pandas 3.0.6 | NumPy 2.5.4


## 1. Carga del Dataset

El archivo pesa ~2.84 GB. Usamos `dtype` optimizados y medimos el tiempo de carga.

In [ ]:
CSV_PATH = Path(r"C:\Users\Erik\Documents\EDA-TF\EDA_2026_2\data\bronze\US_Accidents_March23.csv")

# Tipos optimizados para reducir uso de memoria
DTYPES = {
    'ID':                 'string',
    'Source':             'string',
    'Severity':           'Int8',
    'Start_Time':         'string',
    'End_Time':           'string',
    'Start_Lat':          'Float32',
    'Start_Lng':          'Float32',
    'Distance(mi)':       'Float32',
    'State':              'string',
    'County':             'string',
    'City':               'string',
    'Weather_Condition':  'string',
    'Temperature(F)':     'Float32',
    'Humidity(%)':        'Float32',
    'Visibility(mi)':     'Float32',
    'Wind_Speed(mph)':    'Float32',
    'Junction':           'boolean',
    'Crossing':           'boolean',
    'Traffic_Signal':     'boolean',
    'Stop':               'boolean',
    'Railway':            'boolean',
    'Roundabout':         'boolean',
    'Traffic_Calming':    'boolean',
    'Sunrise_Sunset':     'string',
}

print(f'⏳ Cargando {CSV_PATH.name} ({CSV_PATH.stat().st_size / 1e6:.0f} MB)...')
t0 = time.time()

df = pd.read_csv(
    CSV_PATH,
    dtype=DTYPES,
    low_memory=False
)

elapsed = time.time() - t0
print(f'✅ Cargado en {elapsed:.1f}s')
print(f'   Filas: {df.shape[0]:,}  |  Columnas: {df.shape[1]}')
print(f'   Memoria: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')

⏳ Cargando US_Accidents_March23.csv (3058 MB)...
✅ Cargado en 115.7s
   Filas: 7,728,394  |  Columnas: 46
   Memoria: 10016.7 MB


## 2. Perfilado Inicial

In [3]:
# Vista rápida
df.head(10)

,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),...,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
0,A-1,Source2,3,2016-02-08 05:46:00,2016-02-08 11:00:00,39.865147,-84.058723,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Night,Night,Night
1,A-2,Source2,2,2016-02-08 06:07:59,2016-02-08 06:37:59,39.928059,-82.831184,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Night,Night,Day
2,A-3,Source2,2,2016-02-08 06:49:27,2016-02-08 07:19:27,39.063148,-84.032608,NaN,NaN,0.01,...,False,False,False,False,True,False,Night,Night,Day,Day
3,A-4,Source2,3,2016-02-08 07:23:34,2016-02-08 07:53:34,39.747753,-84.205582,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Day,Day,Day
4,A-5,Source2,2,2016-02-08 07:39:07,2016-02-08 08:09:07,39.627781,-84.188354,NaN,NaN,0.01,...,False,False,False,False,True,False,Day,Day,Day,Day
5,A-6,Source2,3,2016-02-08 07:44:26,2016-02-08 08:14:26,40.10059,-82.925194,NaN,NaN,0.01,...,False,False,False,False,False,False,Day,Day,Day,Day
6,A-7,Source2,2,2016-02-08 07:59:35,2016-02-08 08:29:35,39.758274,-84.230507,NaN,NaN,0.0,...,False,False,False,False,False,False,Day,Day,Day,Day
7,A-8,Source2,3,2016-02-08 07:59:58,2016-02-08 08:29:58,39.770382,-84.194901,NaN,NaN,0.01,...,False,False,False,False,False,False,Day,Day,Day,Day
8,A-9,Source2,2,2016-02-08 08:00:40,2016-02-08 08:30:40,39.778061,-84.172005,NaN,NaN,0.0,...,False,False,False,False,False,False,Day,Day,Day,Day
9,A-10,Source2,3,2016-02-08 08:10:04,2016-02-08 08:40:04,40.10059,-82.925194,NaN,NaN,0.01,...,False,False,False,False,False,False,Day,Day,Day,Day


In [4]:
# Info de tipos y nulos
df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 7728394 entries, 0 to 7728393
Data columns (total 46 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   ID                     string 
 1   Source                 string 
 2   Severity               Int8   
 3   Start_Time             string 
 4   End_Time               string 
 5   Start_Lat              Float32
 6   Start_Lng              Float32
 7   End_Lat                float64
 8   End_Lng                float64
 9   Distance(mi)           Float32
 10  Description            str    
 11  Street                 str    
 12  City                   string 
 13  County                 string 
 14  State                  string 
 15  Zipcode                str    
 16  Country                str    
 17  Timezone               str    
 18  Airport_Code           str    
 19  Weather_Timestamp      str    
 20  Temperature(F)         Float32
 21  Wind_Chill(F)          float64
 22  Humidity(%)            Float3

In [5]:
# Estadísticas descriptivas
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ID,7728394,7728394,A-1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Source,7728394,3,Source1,4325632,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Severity,7728394.0,<NA>,<NA>,<NA>,2.212384,0.487531,1.0,2.0,2.0,2.0,4.0
Start_Time,7728394,6131796,2021-01-26 16:16:13,225,NaN,NaN,NaN,NaN,NaN,NaN,NaN
End_Time,7728394,6705355,2021-11-22 08:00:00,112,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Start_Lat,7728394.0,<NA>,<NA>,<NA>,36.201195,5.076079,24.5548,33.399632,35.823975,40.084957,49.002201
Start_Lng,7728394.0,<NA>,<NA>,<NA>,-94.702553,17.391756,-124.623833,-117.219397,-87.766617,-80.353676,-67.113167
End_Lat,4325632.0,NaN,NaN,NaN,36.261829,5.272905,24.566013,33.46207,36.183495,40.17892,49.075
End_Lng,4325632.0,NaN,NaN,NaN,-95.72557,18.107928,-124.545748,-117.754345,-88.02789,-80.247086,-67.109242
Distance(mi),7728394.0,<NA>,<NA>,<NA>,0.561842,1.776811,0.0,0.0,0.03,0.464,441.75


In [6]:
# Nulos por columna
nulos = df.isnull().sum()
pct_nulos = (nulos / len(df) * 100).round(2)
resumen_nulos = pd.DataFrame({'Nulos': nulos, '% Nulos': pct_nulos})
resumen_nulos[resumen_nulos['Nulos'] > 0].sort_values('% Nulos', ascending=False)

,Nulos,% Nulos
End_Lat,3402762,44.03
End_Lng,3402762,44.03
Precipitation(in),2203586,28.51
Wind_Chill(F),1999019,25.87
Wind_Speed(mph),571233,7.39
Visibility(mi),177098,2.29
Wind_Direction,175206,2.27
Humidity(%),174144,2.25
Weather_Condition,173459,2.24
Temperature(F),163853,2.12
